# 9 · Forced alignment — phone boundaries in time

Stage 6 tells you **what** was said. This stage tells you **when** each phone
starts and ends. The output is a Praat **TextGrid** per utterance, with a word
tier and a phone tier — the format Praat, ELAN and every TTS front-end expect.

The aligner is the Kölsch model from stage 5 plus
`torchaudio.functional.forced_align`. It needs no pronunciation dictionary
beyond `kolsch_g2p.py`, which is the point: MFA and MAUS both need a German
lexicon, and **94.1 % of Kölsch word tokens are out of vocabulary** against the
152,766-word `german_mfa` dictionary.

---

## Start here: `absorb="vc"`

CTC is **peaky**. The model emits one confident frame per phone and blanks in
between, so on the reference material the labelled frames cover only **14–21 %**
of the timeline — the demo below measures its own figure and prints it, and on the shipped one-speaker example it comes out nearer 31 %. Roughly **four fifths of every phone duration you see in a TextGrid is
not measured — it is a rule deciding who gets the blank frames.**

Which rule you pick therefore matters more than the model does, and the obvious
rules are all wrong in the same way. They assume each CTC spike sits in the
middle of its phone. It does not:

| | spike sits … through its segment |
|---|---|
| vowels | **71 %** (late) |
| consonants | **10 %** (early) |

*(measured against MFA over one recording and its two halves, n = 27 vowels /
35 consonants; unstable in detail — the consonant median is 29 % on the full
file and 6 % on the chunks — but the direction is stable)*

So the blank run between a consonant spike and the following vowel spike starts
at the **beginning** of the consonant and ends **two thirds into** the vowel.
Split it down the middle and the consonant eats half the vowel. That is exactly
what you see: `/h/` in *høːt* came out **307 ms** where MFA said 10 ms, and
`/b/` in *bɛsɐ* took **197 ms** while its own vowel kept 42.

`absorb="vc"` gives each word-internal blank run to the **vowel**. Word onsets
are untouched, so cross-system comparisons stay valid.

| gap rule | three-way spread (median) | all three within 50 ms |
|---|---|---|
| `hybrid` | 72.9 ms | 38.2 % |
| **`vc`** | **67.4 ms** | **41.4 %** |

Over 84 field recordings and 3,742 phones: **68 improve, 15 get worse**, median
change −3.0 ms. Read the per-class split before believing the headline — every
vocalic class improves, every consonantal class is flat or very slightly worse.
The two nearly cancel, and "vc is better" is a **net** claim, not a uniform one.


In [ ]:
!pip -q install torch torchaudio transformers librosa soundfile pandas matplotlib
import torch, torchaudio, librosa, soundfile as sf
import numpy as np, pandas as pd
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "· torchaudio", torchaudio.__version__, "·", device)


In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/Koelsch-Phoneme-Recognition.git /content/kolsch-tandem")
except Exception:
    pass

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)
print("repo root:", ROOT)


## The model

Weights are **not** in this repository — the fine-tuned checkpoint is ~1.2 GB,
well past what git should carry. Point `MODEL_DIR` at either

* the folder stage 5 wrote into `models/`, or
* a Hugging Face repo id, once you have pushed the checkpoint there.

`PROCESSOR_DIR` may be the same folder; it is separate here only because the
reference checkpoint keeps the tokenizer beside the weights.


In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# Either a local directory (stage 5 output) or a Hugging Face repo id.
MODEL_DIR     = os.environ.get("KOLSCH_MODEL",     os.path.join(MODELS, "kolsch_wav2vec2_model_all"))
PROCESSOR_DIR = os.environ.get("KOLSCH_PROCESSOR", MODEL_DIR)

processor = Wav2Vec2Processor.from_pretrained(PROCESSOR_DIR)
model     = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).to(device).eval()

VOCAB    = processor.tokenizer.get_vocab()
BLANK_ID = processor.tokenizer.pad_token_id
SEP      = "|"
SAMPLE_RATE = 16_000
print(f"{len(VOCAB)} symbols · blank id {BLANK_ID} · {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")


## Phone classes

`vc` needs to know which symbols are vowels. This table is the model's own
inventory; if you retrain with a different vocabulary, extend it — a symbol
missing here is treated as a **consonant**, which silently disables the rule for
that phone.


In [ ]:
VOWEL_SHORT = set("a e œ ɐ ɔ ə ɛ ɪ ʊ ʏ y".split())
VOWEL_LONG  = set("aː eː iː oː uː yː øː ɛː".split())
DIPHTHONG   = set("aɪ aʊ ɔɪ ɔʏ ɛɪ".split())
PLOSIVE     = set("b d g k p t".split())
AFFRICATE   = set("p͡f t͡s t͡ʃ".split())
FRICATIVE   = set("f h s v z ç ʃ ʒ χ".split())
NASAL       = set("m n ŋ".split())
APPROXIMANT = set("j l ʁ".split())

VOCALIC   = VOWEL_SHORT | VOWEL_LONG | DIPHTHONG
SUSTAINED = FRICATIVE | AFFRICATE          # see the intervocalic exception below

_known = VOCALIC | PLOSIVE | AFFRICATE | FRICATIVE | NASAL | APPROXIMANT
_unclassified = {s for s in VOCAB if s not in _known and s not in {SEP, "[PAD]", "[UNK]", "<s>", "</s>"}}
print("unclassified symbols:", _unclassified or "none")
assert not _unclassified, "classify these before trusting absorb='vc'"


## Text → phone chain

Dictionary first (`data/lexicon.csv`, human-checkable), the rule-based converter
for anything not in it — the same policy as stage 4. The IPA string is then cut
into the model's symbols by longest-match, so `t͡s`, `aɪ` and `øː` stay whole
instead of decomposing into their parts.


In [ ]:
import csv, re
from kolsch_g2p import word_to_ipa

_APOS = "\'\u2019\u02bc\u02bb`"
def _clean(w):
    w = str(w).lower()
    for a in _APOS:
        w = w.replace(a, "")
    return re.sub(r"[^a-zäöüßï]", "", w)

LEX = {}
if os.path.exists(LEXICON):
    for row in csv.DictReader(open(LEXICON, encoding="utf-8")):
        LEX[row["kolsch"]] = row["ipa"]
print(f"{len(LEX)} lexicon entries")

_SYMS = sorted((s for s in VOCAB if s not in {SEP, "[PAD]", "[UNK]", "<s>", "</s>"}),
               key=len, reverse=True)          # longest match first

def ipa_to_phones(ipa):
    """Cut an IPA string into model symbols. Unknown characters are dropped."""
    out, i = [], 0
    while i < len(ipa):
        for s in _SYMS:
            if ipa.startswith(s, i):
                out.append(s); i += len(s); break
        else:
            i += 1
    return out

def text_to_chain(text):
    """-> ([[phones of word 1], ...], [orthographic words])"""
    words, chain = [], []
    for tok in str(text).split():
        w = _clean(tok)
        if not w:
            continue
        phones = ipa_to_phones(LEX.get(w) or word_to_ipa(w))
        if phones:
            words.append(tok); chain.append(phones)
    return chain, words


## The gap rules

Four modes, all operating on the same CTC output. Only the treatment of the
blank runs differs.

| mode | what happens to a blank run |
|---|---|
| `none` | nothing — phones keep only their labelled frames, and the TextGrid has holes |
| `even` | split down the middle |
| `hybrid` | cut at the **spectral-change peak** inside the word; posterior-weighted across word edges |
| **`vc`** | word-internally, C→V and V→C runs go to the **vowel**; everything else as `hybrid` |

**The one exception in `vc`,** and it is fitted rather than derived: an
*intervocalic* fricative or affricate keeps the flux peak. Its left boundary is
already its own spike (from the V→C rule), so giving the right-hand run away too
would leave it a single 20 ms frame — the `/s/` of *bɛsɐ* collapsed to 20 ms
against MFA's 210 and MAUS's 190. A fricative's spike marks the *onset* of
frication and the noise continues; a stop's spike sits at the burst. Phonetics
supports the distinction, **this data does not prove it** — `/s/` in *bɛsɐ* is
the only instance in the material. Revisit if it ever misfires.


In [ ]:
from dataclasses import dataclass

@dataclass
class Segment:
    label: str
    start: float          # in emission frames
    end: float
    score: float

def merge_repeats(tokens, scores, blank_id, tokenizer):
    segs, i, n = [], 0, len(tokens)
    while i < n:
        tok = int(tokens[i]); j = i
        while j < n and int(tokens[j]) == tok:
            j += 1
        if tok != blank_id:
            segs.append(Segment(tokenizer.convert_ids_to_tokens(tok), i, j, scores[i:j].mean().item()))
        i = j
    return segs

def spectral_flux(wav, sr, hop_ms=2.5, n_mfcc=13):
    """Frame-to-frame spectral change: where the signal says a boundary is.

    c0 is dropped so this responds to spectral SHAPE (formant/manner change)
    rather than loudness, which would just track the amplitude envelope.
    """
    hop = max(1, int(hop_ms / 1000 * sr))
    m = librosa.feature.mfcc(y=np.asarray(wav, dtype=np.float32), sr=sr, n_mfcc=n_mfcc,
                             hop_length=hop, n_fft=min(1024, max(256, 4 * hop)))[1:]
    m = (m - m.mean(axis=1, keepdims=True)) / (m.std(axis=1, keepdims=True) + 1e-8)
    flux = np.concatenate([[0.0], np.sqrt((np.diff(m, axis=1) ** 2).sum(axis=0))])
    return flux, np.arange(len(flux)) * hop / sr

def energy_envelope(wav, sr, hop_ms=5.0, win_ms=20.0, smooth=3):
    """Median-smoothed dB energy per frame, and each frame's time."""
    hop, win = max(1, int(hop_ms / 1000 * sr)), max(1, int(win_ms / 1000 * sr))
    n = (len(wav) - win) // hop + 1
    if n < 2:
        return np.zeros(0), np.zeros(0)
    e = np.sqrt(np.array([np.mean(wav[i * hop:i * hop + win] ** 2) for i in range(n)]))
    db = 20 * np.log10(e + 1e-8)
    if smooth > 1:
        k = smooth | 1
        pad = np.pad(db, k // 2, mode="edge")
        db = np.array([np.median(pad[i:i + k]) for i in range(n)])
    return np.arange(n) * hop / sr, db


def pause_span_in_gap(t0, t1, env_t, env_db, thr, min_ms=60.0):
    """The longest sub-threshold run in [t0, t1] as (start, end), or None.

    A word boundary is not one event but three: the previous word ends, there is
    silence, the next word begins. Collapsing them into one cut is what makes the
    phone before a pause absorb the pause.
    """
    if len(env_t) == 0:
        return None
    lo, hi = np.searchsorted(env_t, t0), np.searchsorted(env_t, t1)
    if hi <= lo:
        return None
    quiet, best, run = env_db[lo:hi] < thr, None, None
    for k, q in enumerate(quiet):
        if q:
            run = k if run is None else run
        elif run is not None:
            if best is None or k - run > best[1] - best[0]:
                best = (run, k)
            run = None
    if run is not None and (best is None or len(quiet) - run > best[1] - best[0]):
        best = (run, len(quiet))
    if best is None:
        return None
    a, b = float(env_t[lo + best[0]]), float(env_t[lo + best[1] - 1])
    return (a, b) if (b - a) * 1000 >= min_ms else None


def speech_onset_in_gap(t0, t1, env_t, env_db, thr):
    """Where speech RESUMES in [t0, t1], or None if the gap holds no pause."""
    if len(env_t) == 0:
        return None
    lo, hi = np.searchsorted(env_t, t0), np.searchsorted(env_t, t1)
    if hi <= lo:
        return None
    quiet = np.where(env_db[lo:hi] < thr)[0]
    return float(env_t[lo + quiet[-1]]) if len(quiet) else None


def absorb_gaps(segments, flux, times, sec_per_frame, drop_label=SEP, mode="vc-onset",
                env_t=None, env_db=None, env_thr=None):
    """Hand every blank run to a neighbour. See the table above for the modes."""
    word_internal_only = mode in ("hybrid", "vc", "vc-onset", "vc-sil")
    class_split        = mode in ("vc", "vc-onset", "vc-sil")

    spans_sep = set()
    if word_internal_only:                 # which gaps cross a word boundary
        k = -1
        for seg in segments:
            if seg.label == drop_label:
                spans_sep.add(k)
            else:
                k += 1

    kept = [s for s in segments if s.label != drop_label]
    if not kept:
        return []

    bounds = [float(kept[0].start)]
    sil = {}     # index -> (silence start, silence end), in frames
    for i in range(len(kept) - 1):
        a, b = kept[i], kept[i + 1]
        if b.start <= a.end:
            bounds.append(float(a.end)); continue
        gap = b.start - a.end

        if mode == "none":
            bounds.append(float(a.end)); continue
        if mode == "even":
            bounds.append(a.end + gap / 2); continue
        if i in spans_sep:                  # across a word edge
            if mode in ("vc-onset", "vc-sil") and env_t is not None:
                span = pause_span_in_gap(a.end * sec_per_frame,
                                         b.start * sec_per_frame,
                                         env_t, env_db, env_thr)
                if span is not None:
                    if mode == "vc-sil":
                        sil[i] = (span[0] / sec_per_frame, span[1] / sec_per_frame)
                    bounds.append(span[1] / sec_per_frame); continue
                # The gap usually contains a real pause: put the boundary at the
                # end of it. Falls back to the flux peak when the words run
                # together -- NOT to the class rule, which was tried and turned
                # out to be a one-speaker artefact.
                t = speech_onset_in_gap(a.end * sec_per_frame, b.start * sec_per_frame,
                                        env_t, env_db, env_thr)
                if t is not None:
                    bounds.append(t / sec_per_frame); continue
                t0, t1 = a.end * sec_per_frame, b.start * sec_per_frame
                lo, hi = np.searchsorted(times, t0), np.searchsorted(times, t1)
                if hi > lo:
                    bounds.append(times[lo + int(np.argmax(flux[lo:hi]))] / sec_per_frame)
                    continue
            total = a.score + b.score
            bounds.append(a.end + gap * (a.score / total if total > 0.5 else 1))
            continue
        if class_split:
            va, vb = a.label in VOCALIC, b.label in VOCALIC
            if va and not vb:
                bounds.append(b.start); continue                    # V->C
            if vb and not va and not (a.label in SUSTAINED and i and
                                      kept[i - 1].label in VOCALIC):
                bounds.append(a.end); continue                      # C->V
        t0, t1 = a.end * sec_per_frame, b.start * sec_per_frame      # flux peak
        lo, hi = np.searchsorted(times, t0), np.searchsorted(times, t1)
        bounds.append(times[lo + int(np.argmax(flux[lo:hi]))] / sec_per_frame
                      if hi > lo else float(a.end + gap / 2))
    bounds.append(float(kept[-1].end))

    if mode == "none":                      # keep the holes visible, do not fake them
        return [Segment(s.label, float(s.start), float(s.end), s.score) for s in kept]
    # A phone abutting a detected pause stops at it; the next starts after it.
    # The interval between belongs to nobody and becomes an empty TextGrid
    # interval -- which is what MFA and MAUS emit.
    out = []
    for i, seg in enumerate(kept):
        st = sil[i - 1][1] if (i - 1) in sil else bounds[i]
        en = sil[i][0] if i in sil else bounds[i + 1]
        out.append(Segment(seg.label, st, max(st, en), seg.score))
    return out


## Aligning one utterance

`torchaudio.functional.forced_align` is a Viterbi pass over the CTC lattice: it
returns the single most likely frame-to-token path **given the phone chain you
supply**. It cannot skip, insert or reorder a phone. If the chain is wrong the
alignment is confidently wrong, which is why stage 4 exists and why the words
come from a transcript rather than from the model's own decode.


In [ ]:
def align(wav_path, text, mode="vc-onset"):
    """-> (phone intervals, word intervals, duration) in SECONDS."""
    wav, _ = librosa.load(wav_path, sr=SAMPLE_RATE)
    dur = len(wav) / SAMPLE_RATE

    chain, words = text_to_chain(text)
    if not chain:
        raise ValueError(f"no phones for {text!r}")
    tokens = []
    for k, w in enumerate(chain):
        if k:
            tokens.append(SEP)
        tokens.extend(w)
    ids = torch.tensor([[VOCAB[t] for t in tokens]], dtype=torch.int32, device=device)

    with torch.inference_mode():
        logits = model(torch.tensor(wav, device=device)[None]).logits
    logp = torch.log_softmax(logits, dim=-1)
    sec_per_frame = dur / logp.shape[1]

    path, scores = torchaudio.functional.forced_align(logp, ids, blank=BLANK_ID)
    segs = merge_repeats(path[0], scores[0].exp(), BLANK_ID, processor.tokenizer)

    flux, times = spectral_flux(wav, SAMPLE_RATE)
    env_t, env_db, env_thr = None, None, None
    if mode in ("vc-onset", "vc-sil"):
        env_t, env_db = energy_envelope(wav, SAMPLE_RATE)
        if len(env_db):
            floor, peak = np.percentile(env_db, 15), env_db.max()
            env_thr = floor + 0.15 * (peak - floor)
        else:
            env_t = None
    segs = absorb_gaps(segs, flux, times, sec_per_frame, mode=mode,
                       env_t=env_t, env_db=env_db, env_thr=env_thr)

    phones = [{"label": s.label, "start": s.start * sec_per_frame,
               "end": s.end * sec_per_frame, "score": s.score} for s in segs]

    word_iv, i = [], 0                       # regroup phones back into words
    for orth, w in zip(words, chain):
        grp = phones[i:i + len(w)]; i += len(w)
        if grp:
            word_iv.append({"label": orth, "start": grp[0]["start"], "end": grp[-1]["end"]})
    return phones, word_iv, dur


## TextGrid output\n\nPraat short-text format, two interval tiers. Opens in Praat and ELAN unchanged.

In [ ]:
def write_textgrid(path, dur, tiers):
    """tiers: [(name, [{label,start,end}, ...]), ...]"""
    def pad(ivs):
        """Praat requires a gap-free tier, so holes become empty intervals."""
        out, t = [], 0.0
        for iv in ivs:
            if iv["start"] > t + 1e-6:
                out.append({"label": "", "start": t, "end": iv["start"]})
            out.append(iv); t = iv["end"]
        if t < dur - 1e-6:
            out.append({"label": "", "start": t, "end": dur})
        return out

    L = ['File type = "ooTextFile"', 'Object class = "TextGrid"', "",
         "0", f"{dur:.6f}", "<exists>", str(len(tiers))]
    for name, ivs in tiers:
        ivs = pad(ivs)
        L += ['"IntervalTier"', f'"{name}"', "0", f"{dur:.6f}", str(len(ivs))]
        for iv in ivs:
            L += [f'{iv["start"]:.6f}', f'{iv["end"]:.6f}',
                  '"' + str(iv["label"]).replace('"', '""') + '"']
    Path(path).write_text("\n".join(L) + "\n", encoding="utf-8")
    return path


## Demo — what `vc` actually changes

One utterance, aligned four ways. The table below is the same phone chain every
time; only the gap rule differs, so any duration that moves is the rule moving
it, not the model.


In [ ]:
MANIFEST = os.path.join(SEG, "manifest.csv")
assert os.path.exists(MANIFEST), "run notebook 3 first — it writes data/segments/manifest.csv"
man = pd.read_csv(MANIFEST)
print(len(man), "segments")

row = man.iloc[0]
print("text:", row["text"])

runs = {m: align(row["audio_path"], row["text"], mode=m)[0]
        for m in ("none", "even", "hybrid", "vc", "vc-onset", "vc-sil")}

cmp = pd.DataFrame({
    "phone": [p["label"] for p in runs["vc"]],
    **{f"{m} (ms)": [round((p["end"] - p["start"]) * 1000) for p in runs[m]]
       for m in ("none", "even", "hybrid", "vc", "vc-onset", "vc-sil")},
})
cmp["class"] = ["vowel" if p in VOCALIC else "cons." for p in cmp["phone"]]
cmp["vc − hybrid"] = cmp["vc (ms)"] - cmp["hybrid (ms)"]
display(cmp)

lab = sum(p["end"] - p["start"] for p in runs["none"])
tot = runs["vc"][-1]["end"] - runs["vc"][0]["start"]
print(f"\nCTC labelled {lab:.2f}s of a {tot:.2f}s span — {100*lab/tot:.0f}%. "
      f"The other {100-100*lab/tot:.0f}% is the gap rule.")
for c in ("vowel", "cons."):
    d = cmp.loc[cmp["class"] == c, "vc − hybrid"]
    print(f"  {c:6s} n={len(d):3d}  mean change {d.mean():+6.1f} ms")


Vowels should gain and consonants should lose. If that is not what the two lines above say, the phone-class table is wrong for your inventory.

In [ ]:
import matplotlib.pyplot as plt

def draw(ax, phones, y, colour, h=0.42):
    for p in phones:
        ax.add_patch(plt.Rectangle((p["start"], y), p["end"] - p["start"], h,
                                   facecolor=colour, alpha=0.22, edgecolor=colour, lw=0.9))
        ax.annotate(p["label"], ((p["start"] + p["end"]) / 2, y + h / 2),
                    ha="center", va="center", fontsize=8)

wav, _ = librosa.load(row["audio_path"], sr=SAMPLE_RATE)
dur = len(wav) / SAMPLE_RATE
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(np.arange(len(wav)) / SAMPLE_RATE, wav / (np.abs(wav).max() or 1) * 0.4 + 0.55,
        lw=0.4, color="#c8c7c2")
for phones, name, col, y in ((runs["hybrid"], "hybrid", "#9dc2ea", -0.50),
                             (runs["vc"], "vc", "#2a78d6", -1.05)):
    draw(ax, phones, y, col)
    ax.annotate(name, (-0.008, y + 0.21), xycoords=("axes fraction", "data"),
                ha="right", va="center", fontsize=9, fontweight="bold", color=col)
for a, b in zip(runs["hybrid"][1:], runs["vc"][1:]):        # every boundary that moved
    if abs(a["start"] - b["start"]) > 0.005:
        ax.annotate("", xy=(b["start"], -1.05 + 0.42), xytext=(a["start"], -0.50),
                    arrowprops=dict(arrowstyle="->", color="#990011", lw=1.0, alpha=0.75))
ax.set_xlim(0, dur); ax.set_ylim(-1.25, 1.05); ax.set_yticks([])
ax.set_xlabel("time (s)"); ax.set_title(row["text"], loc="left", fontsize=10)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()


## Batch — a TextGrid for every segment

In [ ]:
TG_DIR = os.path.join(DATA, "textgrids"); os.makedirs(TG_DIR, exist_ok=True)
ABSORB = "vc-onset"                             # the recommended default

ok = fail = 0
for _, r in man.iterrows():
    stem = Path(r["audio_path"]).stem
    try:
        phones, words, dur = align(r["audio_path"], r["text"], mode=ABSORB)
        write_textgrid(os.path.join(TG_DIR, f"{stem}.TextGrid"), dur,
                       [("words", words), ("phones", phones)])
        ok += 1
    except Exception as e:
        fail += 1
        print(f"  {stem}: {type(e).__name__}: {e}")
print(f"{ok} TextGrids -> {TG_DIR}" + (f"   ({fail} failed)" if fail else ""))


---

## Comparing against MFA and MAUS

Neither runs here, and both are **optional**. They are documented because the
`vc` numbers quoted at the top are agreement figures against these two, and an
agreement figure you cannot reproduce is not a result.

Give all three systems the **same phone chain** — otherwise you are measuring
three different pronunciation dictionaries, not three aligners.

### MFA (local, offline)

```bash
conda install -c conda-forge montreal-forced-aligner   # 2.2.17 here
mfa model download acoustic german_mfa
mfa model download dictionary german_mfa
mfa align data/segments/ german_mfa german_mfa out/mfa/
```

Runs entirely on your machine. Expect a large OOV rate on Kölsch — 94.1 % of
tokens against the pretrained dictionary — so supply your own lexicon built from
`kolsch_g2p.py` if you want the comparison to be fair.

### MAUS (BAS webservice)

> **This uploads your audio.** MAUS runs at the Bavarian Archive for Speech
> Signals in Munich; the recordings and their transcripts leave your machine. It
> is an academic service, not a commercial one, but it is still a third party.
> For the material in this project that was an acceptable trade for a comparison
> baseline. **For recordings your speakers did not consent to share, it is not.**
> Everything else in this repository runs locally.

```bash
curl -X POST -F SIGNAL=@utt.wav -F BPF=@utt.par -F LANGUAGE=deu-DE -F MODUS=align \
  -F OUTFORMAT=TextGrid \
  https://clarin.phonetik.uni-muenchen.de/BASWebServices/services/runMAUS
```

### Reading the comparison

With no hand-corrected reference, "which aligner is right" is not answerable.
What the numbers above report is the **three-way spread** — how far apart the
three systems place the same boundary — plus each system's deviation from the
median of the three. That median leans toward MFA and MAUS, which share an
HMM-GMM lineage, so a wav2vec2-specific improvement is *understated* by it. Treat
these as agreement, not accuracy, until a hand-corrected subset exists.


In [ ]:
# A run-anywhere check that the recommended default is doing what it claims.
_p, _w, _d = align(man.iloc[0]["audio_path"], man.iloc[0]["text"], mode="vc-onset")
assert _p[0]["start"] >= 0 and abs(_p[-1]["end"] - _w[-1]["end"]) < 1e-6
assert all(b["start"] >= a["end"] - 1e-6 for a, b in zip(_p, _p[1:])), "phones overlap"
_gap = sum(b["start"] - a["end"] for a, b in zip(_p, _p[1:]))
print(f"silence emitted: {_gap*1000:.0f} ms "
      f"(vc emits 0 by construction; vc-sil emits the pauses)")
assert len(_w) == len(str(man.iloc[0]["text"]).split()), "word count drifted"
print(f"OK — {len(_p)} phones, {len(_w)} words, {_d:.2f}s, monotonic and gap-free")
